# 05 · Batch DWH trên stack Docker

Notebook này quan sát **stack thật**: HDFS, HiveServer2, Spark standalone, Parquet partition.
Mọi con số đọc từ artifact do job sinh ra, không nhập tay.

Điều kiện: `python scripts/stack.py up` và `python scripts/stack.py bootstrap` đã chạy xong.

In [1]:
import json
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ART = ROOT / "artifacts"
REPORTS = ROOT / "reports"

def sh(command, timeout=600):
    result = subprocess.run(command, capture_output=True, text=True, timeout=timeout)
    return result.stdout + result.stderr

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
GREEN, TEAL, GOLD, RED = "#1F6B45", "#2B6E7A", "#8F5E1A", "#A23B2A"

## 1. Container nào đang chạy?

In [2]:
services = sh(["docker", "compose", "-f", str(ROOT / "docker" / "docker-compose.yml"), "ps",
              "--format", "{{.Name}}\t{{.Status}}"])
print("\n".join(sorted(services.strip().splitlines())))

agri-exporter	Up 7 hours
agri-postgres	Up 7 hours (healthy)
agri-tools	Up 6 hours
airflow	Up 6 hours
datanode	Up 17 minutes (healthy)
grafana	Up 7 hours
hive-metastore	Up 7 hours
hiveserver2	Up 6 minutes
kafka	Up 17 minutes
kafka-exporter	Up 7 hours
namenode	Up 48 minutes (healthy)
prometheus	Up 7 hours
spark-master	Up 7 hours
spark-worker	Up 7 hours
superset	Up 6 hours (healthy)


## 2. Dữ liệu thô đã nằm trong HDFS

In [3]:
print(sh(["docker", "exec", "namenode", "bash", "-c", "hdfs dfs -ls -R /data/agri/raw | head -20"]))

drwxr-xr-x   - root supergroup          0 2026-11-12 03:26 /data/agri/raw/daily
-rw-r--r--   1 root supergroup    9903845 2026-11-12 03:26 /data/agri/raw/daily/fact_weather_daily.csv
drwxr-xr-x   - root supergroup          0 2026-11-12 03:26 /data/agri/raw/hourly
-rw-r--r--   1 root supergroup   61272107 2026-11-12 03:26 /data/agri/raw/hourly/fact_weather_hourly.csv
drwxr-xr-x   - root supergroup          0 2026-11-12 03:26 /data/agri/raw/location
-rw-r--r--   1 root supergroup       1769 2026-11-12 03:26 /data/agri/raw/location/dim_location.csv
drwxr-xr-x   - root supergroup          0 2026-11-12 03:26 /data/agri/raw/yield
-rw-r--r--   1 root supergroup      21798 2026-11-12 03:26 /data/agri/raw/yield/fact_cereal_annual.csv



## 3. Hive thấy những bảng và view nào?

In [4]:
print(sh(["docker", "exec", "hiveserver2", "beeline", "-u",
          "jdbc:hive2://localhost:10000/;auth=noSasl", "-n", "hive",
          "-e", "SHOW TABLES;"], timeout=180))

+-----------+
| tab_name  |
+-----------+
+-----------+
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000/;auth=noSasl
Connected to: Apache Hive (version 2.3.2)
Driver: Hive JDBC (version 2.3.2)
Transaction isolation: TRANSACTION_REPEATABLE_READ
No rows selected (0.233 seconds)
Beeline version 2.3.2 by Apache Hive
Closing: 0: jdbc:hive2://localhost:10000/;auth=noSasl



## 4. Cổng chất lượng dữ liệu (12 kiểm tra)

In [5]:
quality = read_json(ART / "quality" / "quality_report.json")
rows = [{"check": c["check"], "value": c["value"], "passed": c["passed"]} for c in quality.get("checks", [])]
display(pd.DataFrame(rows))
print("PASS" if quality.get("passed") else "FAIL", "- failed:", quality.get("failed"))

,check,value,passed
0,hourly_rows,1052064,True
1,duplicate_keys,0,True
2,null_core_fields,0,True
3,temperature_range,0,True
4,soil_range,0,True
5,hourly_locations,12,True
6,partitions,120,True
7,view_join,1052064,True
8,daily_rows,175328,True
9,locations,32,True


PASS - failed: []


## 5. Sản lượng của từng job Spark

In [6]:
metrics = {}
for path in sorted((ART / "metrics").glob("*.json")):
    metrics[path.stem] = read_json(path)
flat = []
for name, payload in metrics.items():
    if isinstance(payload, dict) and ("rows" in payload or "locations" in payload):
        flat.append({"job": name, **{k: payload[k] for k in ("rows", "locations", "partitions", "countries", "years") if k in payload}})
display(pd.DataFrame(flat))

,job,rows,locations,partitions,countries,years
0,01_clean_weather,1052064.0,12.0,120.0,NaN,NaN
1,02_build_daily,175328.0,32.0,NaN,NaN,NaN
2,03_build_dimensions,NaN,32.0,NaN,NaN,NaN
3,04_load_yield,450.0,NaN,NaN,30.0,15.0
4,06_build_features,450.0,NaN,NaN,30.0,15.0
5,09_bench_variant,1052064.0,NaN,NaN,NaN,NaN
6,stream_restart_test,100000.0,NaN,NaN,NaN,NaN
7,stream_sink,100000.0,NaN,NaN,NaN,NaN


## 6. Benchmark Hive vs Spark SQL

In [7]:
bench = ART / "bench" / "fullstack_benchmark.csv"
if bench.exists():
    frame = pd.read_csv(bench)
    display(frame.sort_values(["engine", "source", "scale", "query"]).head(40))
    pivot = frame[frame["scale"] == frame["scale"].max()].pivot_table(
        index="query", columns="engine", values="seconds_median", aggfunc="median")
    display(pivot)
else:
    print("Chưa có benchmark. Chạy: python scripts/stack.py benchmark")

,engine,source,scale,query,seconds_median,result_rows
2,hive,parquet_partitioned,100000,avg_soil_when_hot,19.811,12
1,hive,parquet_partitioned,100000,avg_yield_by_location,20.625,12
0,hive,parquet_partitioned,100000,count_all,19.494,1
3,hive,parquet_partitioned,100000,top10_yield,20.422,10
6,hive,parquet_partitioned,1052064,avg_soil_when_hot,24.144,12
5,hive,parquet_partitioned,1052064,avg_yield_by_location,18.208,12
4,hive,parquet_partitioned,1052064,count_all,17.711,1
7,hive,parquet_partitioned,1052064,top10_yield,31.773,10
26,spark_sql,csv,100000,avg_soil_when_hot,1.129,2
25,spark_sql,csv,100000,avg_yield_by_location,1.253,2


engine,hive,spark_sql
query,,
avg_soil_when_hot,24.144,2.531
avg_yield_by_location,18.208,2.161
count_all,17.711,2.170
top10_yield,31.773,2.594


## Nghiệm thu P1

- Đúng **1.052.064** dòng hourly trong Hive, 120 partition năm-tháng.
- 0 null ở ba biến chính, 0 duplicate theo `(location_id, event_time)`.
- 4 truy vấn benchmark trả kết quả trên cả 100K và 1M dòng.
- Chạy lại pipeline không nhân đôi dữ liệu (job dùng overwrite + drop table).